In [1]:
import pandas as pd
import os

In [4]:
customer_file_path = 'data/customers.csv'

file_size_bytes = os.path.getsize(customer_file_path)

print(f"File: {customer_file_path}")
print(f"File size: {file_size_bytes}")

df = pd.read_csv(customer_file_path)
rows, cols = df.shape
print(f"Row Count: {rows}")
print(f"Column Count: {cols}\n")

print("--- Columns & Inferred Types ---")
print(df.dtypes)
print()

print("--- Missing Values ---")
print(df.isnull().sum())
print()

duplicate_rows = df.duplicated().sum()
print("--- Duplicates ---")
print(f"Exact Duplicate Rows: {duplicate_rows}\n")

print("--- Unique Constraints ---")
if 'customer_id' in df.columns:
    is_unique = df['customer_id'].is_unique
    print(f"Is 'customer_id' unique? {is_unique}")
else:
    print("'customer_id' column not found.")


File: data/customers.csv
File size: 18183
Row Count: 250
Column Count: 7

--- Columns & Inferred Types ---
customer_id         str
first_name          str
last_name           str
email               str
city                str
signup_date         str
customer_segment    str
dtype: object

--- Missing Values ---
customer_id         0
first_name          0
last_name           0
email               3
city                2
signup_date         0
customer_segment    0
dtype: int64

--- Duplicates ---
Exact Duplicate Rows: 2

--- Unique Constraints ---
Is 'customer_id' unique? False


In [ ]:
import json
import pandas as pd

file_path = 'data/orders.json'

with open(file_path, 'r') as f:
    data = json.load(f)

# 1. Confirm root structure
is_list = isinstance(data, list)
print("--- Root Structure ---")
print(f"Is root structure a list of records? {is_list}\n")

if is_list and len(data) > 0:
    sample_record = data[0]
    top_level_keys = list(sample_record.keys())
    nested_fields = [k for k, v in sample_record.items() if isinstance(v, dict)]
    
    print("--- Keys & Structure ---")
    print(f"Top-level keys: {top_level_keys}")
    print(f"Nested fields: {nested_fields}\n")

    df = pd.DataFrame(data)

    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = pd.to_datetime(df[col])
            except (ValueError, TypeError):
                pass

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    timestamp_cols = df.select_dtypes(include=['datetime']).columns.tolist()
    
    print("--- Data Types ---")
    print(f"Numeric fields: {numeric_cols}")
    print(f"Timestamp fields: {timestamp_cols}\n")

    print("--- Record Count & Missing Values ---")
    print(f"Total Records: {len(df)}")
    print("Nulls/Missing Keys per Top-Level Column:")
    print(df.isnull().sum())
else:
    print("File is empty or not a list of records.")

--- Root Structure ---
Is root structure a list of records? True

--- Keys & Structure ---
Top-level keys: ['order_id', 'customer_id', 'order_timestamp', 'status', 'item_count', 'subtotal', 'shipping_fee', 'total_amount', 'shipping']
Nested fields: ['shipping']

--- Data Types ---
Numeric fields: ['item_count', 'subtotal', 'shipping_fee', 'total_amount']
Timestamp fields: []

--- Record Count & Missing Values ---
Total Records: 250
Nulls/Missing Keys per Top-Level Column:
order_id           0
customer_id        0
order_timestamp    0
status             0
item_count         0
subtotal           0
shipping_fee       0
total_amount       0
shipping           0
dtype: int64


In [6]:
pq_path = 'data/products.parquet'
csv_path = 'data/products.csv'
json_path = 'data/products.json'

df_pq = pd.read_parquet(pq_path)

print("--- Parquet Profile ---")
print(f"File Size: {os.path.getsize(pq_path) / 1024:.2f} KB")
print(f"Shape: {df_pq.shape}")
print(f"Dtypes:\n{df_pq.dtypes}\n")

if os.path.exists(csv_path) and os.path.exists(json_path):
    df_csv = pd.read_csv(csv_path)
    df_json = pd.read_json(json_path)
    
    print("--- File Size Comparison ---")
    print(f"Parquet: {os.path.getsize(pq_path) / 1024:.2f} KB")
    print(f"CSV:     {os.path.getsize(csv_path) / 1024:.2f} KB")
    print(f"JSON:    {os.path.getsize(json_path) / 1024:.2f} KB\n")
    
    print("--- Type Preservation Comparison ---")
    date_col = 'launch_date' if 'launch_date' in df_pq.columns else df_pq.columns[0]
    print(f"Parquet type for '{date_col}': {df_pq[date_col].dtype}")
    print(f"CSV type for '{date_col}':     {df_csv[date_col].dtype}")
    print(f"JSON type for '{date_col}':    {df_json[date_col].dtype}")

--- Parquet Profile ---
File Size: 14.31 KB
Shape: (200, 7)
Dtypes:
product_id            str
product_name          str
category              str
brand                 str
unit_price        float64
stock_quantity      int32
weight_kg         float64
dtype: object



In [7]:
import requests

base_url = 'http://127.0.0.1:8000/api/events'

# Fetch Page 1
response_p1 = requests.get(f"{base_url}?page=1&per_page=10").json()
print("--- PAGE 1 ---")
print(f"Current Page: {response_p1['page']}")
print(f"Items retrieved: {len(response_p1['items'])}")
print(f"Next page to call: {response_p1['next_page']}\n")

next_page_num = response_p1['next_page']
response_p2 = requests.get(f"{base_url}?page={next_page_num}&per_page=10").json()

print("--- PAGE 2 ---")
print(f"Current Page: {response_p2['page']}")
print(f"Items retrieved: {len(response_p2['items'])}")
print(f"Has more data? {response_p2['has_more']}")

--- PAGE 1 ---
Current Page: 1
Items retrieved: 10
Next page to call: 2

--- PAGE 2 ---
Current Page: 2
Items retrieved: 10
Has more data? True
